# Many Strikes, Many adjustments

### Helper funcs

In [ ]:
def w_sum(a, b):
    assert len(a) == len(b)
    output = 0
    for i in range(len(a)):
        output += a[i] * b[i] # multiply and accumulate
    return output



def vect_mat_mul(vector, matrix):
    output = [0] * len(matrix)
    for i in range(len(matrix)):
        output[i] = w_sum(vector,
                          matrix[i])
    return output


### Part 1: Multi Input - Kevin

In [ ]:
# week4/part1_multi_input.py
import numpy as np

# No Numpy version

# the three inputs for sensing 0: blade_angle, balance, breath
sensing_0 = [8.5, 0.65, 1.2]
weights = [0.1, 0.2, -0.1]
goal = 1.0
alpha = 0.01
iterations = 10

def w_sum(a, b):
    # multiplies each pair of numbers and adds them all up (dot product)
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def ele_mul(scalar, vector):
    # multiplies one number by every item in a list
    output = []
    for i in range(len(vector)):
        output.append(scalar * vector[i])
    return output

def gradient_descent_multi(input, weights, true, alpha, iterations):
    # keeps its own copy so it doesn't change the list we were handed
    weights = list(weights)

    error_history = []
    weight_history = []

    for iteration in range(iterations):
        # predict: combine the 3 inputs and 3 weights into one number
        pred = w_sum(input, weights)

        # compare: how far off we are, and in which direction
        error = (pred - true) ** 2
        delta = pred - true

        # learn: the same delta gets spread across the 3 weights,
        # scaled by each weight's own input
        weight_deltas = ele_mul(delta, input)
        for i in range(len(weights)):
            weights[i] -= alpha * weight_deltas[i]

        # save a copy, not the live list, or every entry would end up
        # pointing at the same final weights
        error_history.append(error)
        weight_history.append(list(weights))

        print(f"Iteration {iteration}: pred={pred:.5f}  error={error:.6f}  weights={[round(w,5) for w in weights]}")

    return weights, error_history, weight_history

# Numpy Version

def gradient_descent_multi_np(input, weights, true, alpha, iterations):
    # convert to arrays so numpy can do the math elementwise
    input = np.array(input)
    weights = np.array(weights, dtype=float)

    error_history = []
    weight_history = []

    for iteration in range(iterations):
        # np.dot does the same job as our w_sum
        pred = np.dot(input, weights)
        error = (pred - true) ** 2
        delta = pred - true

        # numpy multiplies delta by every input automatically, no loop needed
        weight_deltas = delta * input
        weights = weights - alpha * weight_deltas

        error_history.append(error)
        weight_history.append(weights.copy())

    return weights, error_history, weight_history

# Verify both versions agree

if __name__ == '__main__':

    plain_weights, plain_errors, plain_hist = gradient_descent_multi(sensing_0, weights, goal, alpha, iterations)
    numpy_weights, numpy_errors, numpy_hist = gradient_descent_multi_np(sensing_0, weights, goal, alpha, iterations)

    print("\nFinal weights (plain vs numpy):")
    for i in range(len(plain_weights)):
        a = plain_weights[i]
        b = numpy_weights[i]
        # abs(a - b) < 1e-10 checks "close enough" instead of exact equality
        print(f"weight {i}: plain={a:.8f}   numpy={b:.8f}   match={abs(a - b) < 1e-10}")

'''
Which weight changed the most / least, and why:
weight 0 (blade_angle) moves the most, weight 1 (balance) moves the least. 
weight_delta = delta * input, and delta is the same shared number for all three weights since they come from one prediction.
The only thing that differs between weights is their own input value, so the weight attached to the biggest input (blade_angle = 8.5)
 gets the biggest push every iteration, and the one attached to the smallest input (balance = 0.65) gets pushed the least.

'''


### Part 1b: Rescaled Inputs - Kevin

In [ ]:
# import from the part1
from part1_multi_input import gradient_descent_multi

# the four sparring sensings, same as previous weeks
blade_angle = [8.5, 9.5, 9.9, 9.0]
balance = [0.65, 0.80, 0.80, 0.90]
breath = [1.2, 1.3, 0.5, 1.0]

def normalize(channel):
    # divide every value by the biggest value in that same list
    biggest = max(channel)
    return [value / biggest for value in channel]

# scale each channel against itself, never against another channel
blade_angle_norm = normalize(blade_angle)
balance_norm = normalize(balance)
breath_norm = normalize(breath)

print("blade_angle normalized:", blade_angle_norm)
print("balance normalized:    ", balance_norm)
print("breath normalized:     ", breath_norm)

# rebuild sensing 0 out of the normalized channels
sensing_0_scaled = [blade_angle_norm[0], balance_norm[0], breath_norm[0]]
print("\nsensing 0 raw:   ", [8.5, 0.65, 1.2])
print("sensing 0 scaled:", sensing_0_scaled)

starting_weights = [0.1, 0.2, -0.1]
goal = 1.0

# raw run, same as Part 1: alpha = 0.01
print("\n--- raw run, alpha=0.01 ---")
raw_weights, raw_errors, raw_hist = gradient_descent_multi(
    [8.5, 0.65, 1.2], starting_weights, goal, 0.01, 20
)

# scaled run, unchanged function, just bigger alpha since inputs are small now
print("\n--- scaled run, alpha=0.1 ---")
scaled_weights, scaled_errors, scaled_hist = gradient_descent_multi(
    sensing_0_scaled, starting_weights, goal, 0.1, 20
)

# print both error histories side by side
print("\nError comparison (raw vs scaled):")
print(f"{'iter':<6}{'raw error':<15}{'scaled error':<15}")
for i in range(len(raw_errors)):
    print(f"{i:<6}{raw_errors[i]:<15.8f}{scaled_errors[i]:<15.8f}")

'''
(a) Raw inputs at alpha=0.1 diverge, scaled inputs don't. Why:
weight_delta = delta * input. blade_angle = 8.5 is a big number, so any
delta gets multiplied into a big weight_delta, and with alpha=0.1 that
push is too large -- the weight overshoots the goal, which makes the next
delta bigger and flipped in sign, and it spirals out of control. Once every
input is scaled down to between 0 and 1, the same alpha=0.1 produces a
small weight_delta, so the updates stay controlled and the error goes down
steadily instead of exploding.

(b) The scaled run's error is bigger than the raw run's error at every
iteration, and that's not a regression. The two runs aren't measuring the
same thing -- the raw run's error is already tiny because its inputs and
weights happen to sit close to a solution, while the scaled run started
further from its solution in the new, smaller-scale space. What actually
matters when comparing runs is how fast and how stably each one converges,
not the raw error number at some fixed iteration, since that number
depends on the scale of the data.

(c) Min-max normalization is (value - min) / (max - min). For blade_angle,
the min is 8.5 (which is sensing 0's own value) and the max is 9.9, so:
(8.5 - 8.5) / (9.9 - 8.5) = 0 / 1.4 = 0
sensing 0's blade_angle becomes exactly 0. Since weight_delta = delta *
input, a 0 input means that weight's delta is 0 too, no matter what the
error is -- that weight stops updating completely. That's the same thing
that happens in Part 4 when a weight is frozen on purpose.
'''

### Part 2: Multi Output - Colson

In [ ]:
import numpy as np


def gradient_descent_outputs(input, weights, trues, alpha, iterations):
    error_history = []
    weight_history = []

    for iteration in range(iterations):
        # Make predictions
        predictions = [input * weight for weight in weights]

        # Calculate deltas
        deltas = [predictions[i] - trues[i] for i in range(3)]

        # Calculate the Mean Squared Error
        errors = [d ** 2 for d in deltas]
        mse = sum(errors) / len(errors)

        error_history.append(mse)
        weight_history.append(weights.copy())

        # Calculate weight detlas
        weight_deltas = [delta * input for delta in deltas]
        for i in range(3):
            weights[i] -= alpha * weight_deltas[i]

        # Print stuff 
        print("Iteration: ", (iteration + 1))
        print("Predictions: ", [round(p, 4) for p in predictions])
        print("Mean Squared Error: ", round(mse, 4))
        print("Weights: ", [round(w, 4) for w in weights])
        print()

    return weights, error_history, weight_history


# -- NumPy Version --
def gradient_descent_outputs_numpy(input, weights, trues, alpha, iterations):
    weights = np.array(weights)
    trues = np.array(trues)

    error_history = []
    weight_history = []

    for iteration in range(iterations):
        predictions = input * weights

        deltas = predictions - trues

        errors = deltas ** 2
        mse = np.mean(errors)

        error_history.append(mse)
        weight_history.append(weights.copy())

        weight_deltas = deltas * input

        weights -= alpha * weight_deltas

    return weights, error_history, weight_history


def main():
    weights = [0.3, 0.2, 0.9]
    input = .65
    trues_multi = [0, 1, 0]
    alpha = .1

    final_weights, error_history, weight_history = gradient_descent_outputs(input, weights, trues_multi, alpha, 20)

    numpy_weights, numpy_errors, numpy_weight_history = (gradient_descent_outputs_numpy(input, [.3, .2, .9], trues_multi, alpha, 20))

    print("Numpy final weights: ", numpy_weights)
    print("Final weights match: ", np.allclose(final_weights, numpy_weights))
    print("Error histories match: ", np.allclose(error_history, numpy_errors))
    print("Weight histories match: ", np.allclose(weight_history, numpy_weight_history))


if __name__ == "__main__":
    main()


# Yes all three shrink by the same factor because they are all using the same input and alpha.
# The first output ended closest to its target of 0 at about .0859. No I don't believe this
# means it converged faster, because although it ended closest to its target, they all changed at
# the same ratio so it only ended up closest because it started closest. Output 2 was still fairly
# far off from its target, which means a single sensing doesn't always mean it reaches its target
# quickly, but if you kept doing iterations it would keep reducing the error.

### Part 3: Multi Input and Output - Brady

In [ ]:
import numpy as np
from helpers import vect_mat_mul


# Calculates the outer results of the two lists and creates a matrix with Len(a) and Len(b) columns
def outer_prod(a, b):
    result = []

    for i in range(len(a)):
        row = []

        for j in range(len(b)):
            row.append(a[i] * b[j])

        result.append(row)

    return result


# Performs Gradient descent for multiple inputs and outputs. The weights are stored in a 3x3 matrix
def gradient_descent_full(input, weights, trues, alpha, iterations):
    weights = [row[:] for row in weights]

  # Stores the result from each iteration
    error_history = []
    weight_history = []
    prediction_history = []
    
  # Repeat the gradient descent process for the requested number of iteration
    for i in range(iterations):
        predictions = vect_mat_mul(input, weights)
        prediction_history.append(predictions[:])

        deltas = []
        squared_errors = []
        
        for j in range(len(predictions)):
            delta = predictions[j] - trues[j]
            deltas.append(delta)
            squared_errors.append(delta ** 2)

        error = sum(squared_errors) / len(squared_errors)
        error_history.append(error)

        weight_deltas = outer_prod(deltas, input)
        # Update every weight using the learning rate (alpha)
        for r in range(len(weights)):
            for c in range(len(weights[r])):
                weights[r][c] = weights[r][c] - alpha * weight_deltas[r][c]
        # Saves a copy of the weight after the iteration
        weight_snapshot = [row[:] for row in weights]
        weight_history.append(weight_snapshot)
    # Returns the final weights and the histories from training
    return weights, error_history, weight_history, prediction_history

# This is the Numpy version of the gradient descent function
def gradient_descent_full_numpy(input, weights, trues, alpha, iterations):
  # Converts the list into Numpy arrays
  input = np.array(input, dtype=float)
  weights = np.array(weights, dtype=float)
  trues = np.array(trues, dtype=float)

  # Stores the reults from each iteration
  error_history = []
  weight_history = []
  prediction_history = []

  for i in range(iterations):
    # Numpy will perform the matrix multiplication for the predictions
    predictions = np.dot(input, weights)
    prediction_history.append(predictions.copy())

    # Calculate the output deltas and mean square error
    deltas = predictions - trues
    error = np.mean(deltas ** 2)
    error_history.append(error)

    weight_deltas = np.outer(deltas, input)
    weights = weights - alpha * weight_deltas

    # Saves the weights from thsi iteration
    weight_history.append(weights.copy())

  return weights, error_history, weight_history, prediction_history

# This section will only run when this file is run directly
if __name__ == '__main__':
  # This is the starting matrix
  weights = [[0.1, 0.1, -0.3],
             [0.1, 0.2, 0.0],
             [0.0, 1.3, 0.1]]

  # Input values for the first sensing
  input = [8.5, 0.65, 1.2]
  
  # True target values for the three outputs
  trues = [0.0, 1.0, 0.0]

  # Train the scratch model for 15 iterations with alpha being 0.01
  final_weights, error_history, weight_history, prediction_history = gradient_descent_full( input, weights, trues, 0.01, 15 )

  # Print the prediction, error, and weights from each iteration
  print("\nIteration log:")

  for i in range(15):
    print( 
      f"Iteration {i + 1}: "
      f"Prediction = {prediction_history[i]} | "
      f"Error = {error_history[i]:.4f} | "
      f"Weights = {weight_history[i]}"
    )
        
  # Show the finals weights with it comparing the first and last errors
  print("Final weights:")
  print(final_weights)

  print("Initial error:", error_history[0])
  print("Final error:", error_history[-1])


  # Run the same experiment using Numpy
  numpy_weights, numpy_errors, numpy_weight_history, numpy_predictions = gradient_descent_full_numpy( input, weights, trues, 0.01, 15)

  print("\nNumpy final weights:")
  print(numpy_weights)

  # Check if the Numpy versions agree
  print("\nParity check:")
  print("Weights match:", np.allclose(final_weights, numpy_weights))
  print("Errors match:", np.allclose(error_history, numpy_errors))

  # Compare the final weights to the original starting weights 
  print("\nWeight movement:")

  initial_weights = [[0.1, 0.1, -0.3],
                     [0.1, 0.2, 0.0],
                     [0.0, 1.3, 0.1]]

  for r in range(len(final_weights)):
    for c in range(len(final_weights[r])):
      movement = abs(final_weights[r][c] - initial_weights[r][c])
      print(f"Weight [{r}][{c}] moved by {movement:.6f}")

# Weight movement depends on both the output delta and the input value.
# Each weight is corrected using:
# weight_delta[i][j] = delta[i] * input[j]
# Within one row, larger input values cause larger weight movement.
# between differemt rows, a larger output delta causes more movement
# meaning its prediction is already close to the true target
                    


### Part 4: Frozen Weights - Elijah

In [ ]:
from part1_multi_input import ele_mul, w_sum

def gradient_descent_frozen(input, weights, true, alpha, iterations, frozen):
    # initialize error and weight history lists
    error_history = []
    weight_history = []
    ws = weights.copy()

    # For every iteration:
    for i in range(iterations):
        #append current weights
        weight_history.append(ws.copy())


        #calculate preds
        pred = w_sum(input, ws)

        #calculate delta and errors (pred - true) = delta
        delta = pred - true
        error = delta ** 2

        #add the error into the error history
        error_history.append(error)

        #Get weight_deltas by multiplying the delta * every input and storing in list
        weight_deltas = ele_mul(delta, input)

        #Frozen is a list of indices for weights that are to remain unchanged
        for index in frozen:
            #By setting the weight_deltas at that index to zero, the weight will not change
            #   weight - alpha(0) = weight
            weight_deltas[index] = 0

        #Update the weight vector by subtracting the corresponding weight delta * alpha for every weight
        for j in range(len(ws)):
            ws[j] = ws[j] - alpha * weight_deltas[j]

    final_weights = ws

    return final_weights, error_history, weight_history


# blade, balance, breath
sensing = [8.5, .65, 1.2]
starting_ws = [.1, .2, -.1]
true = 1

#For each test, these will be the different weights frozen, and the alphas for the corresponding test
frozen_indices = [[], [0, 2], [0,1]]
alphas = [.01, .3, .3]

iters = 5


for i in range(3):
    final_ws, errors, weights, = gradient_descent_frozen(sensing, starting_ws, true, alphas[i], iters, frozen_indices[i])

    print(f"\nTest {i+1}:")
    print(f"Final weights = {final_ws}")
    print(f"Weight history = {weights}")
    print(f"Errors = {errors}")


"""
In run 2, the balance weight is doing all of the correcting. In run 3, it is breath that does this.
When a weight is frozen, its point on the error curve is frozen, so the only way to move down the bowl
is for the other weights to translate the bowl towards the frozen weight, rather than moving the weight down
the bowl.

For run 1, S = 8.5^2 + .65^2 + 1.2^2 = 74.1125. 1 - .01*74.1125 = .258875
Run 2: S = .65^2 = .4225.    1 - .3 * .4225 = .87325
Run 3: S = 1.2^2 = 1.44.   1 -.3 *1.44 = .568

When blade angle is frozen, S is decreased by a very large amount (8.5^2 = 72.25). The miss must be multiplied by a number between
1 and -1, so 1 - alpha * S must be between 1 and -1. When S is very high (72.25) alpha must be much smaller to keep the miss from
being too large in magnitude, specifically alpha * S must be between 0 and 2. Alpha must be more than 0 and less than = 2/S.
2/74.1125 (S of first run) = ~.027, so alpha must be less than that. For the the other two runs, the greatest S = 1.44,
2/1.44 = ~ 1.389 is the max alpha for that run. So, the alpha can be much larger when your highest changeable inputs are much smaller.
"""




### Part 5: Visualizing - Robert

In [ ]:
import matplotlib
matplotlib.use("Agg") # render to a file; no display needed
import matplotlib.pyplot as plt

# training functions
from part1_multi_input import gradient_descent_multi
from part4_freeze import gradient_descent_frozen


## Figure 1 - Part 1's weights over time

# senses index 0 and alpha 0.01, for at least 10 iterations
weights, errors, weight_history = gradient_descent_multi(
    [8.5, 0.65, 1.2], [0.1, 0.2, -0.1], 1.0, 0.01, 10) # sensing 0, weights, goal, alpha, iterations

for j, name in enumerate(["blade_angle", "balance", "breath"]):
    plt.plot([w[j] for w in weight_history], label=name)

plt.xlabel("iteration")
plt.ylabel("weight value")
plt.title("Part 1 weights, alpha = 0.01")
plt.legend()
plt.savefig("week4/fig1_weights.png", dpi=150)
plt.close() # start a clean figure for the next plot

## Figure 2 - Part 4's Frozen vs Unfrozen function

# Unfrozen run
base_w, base_err, base_hist = gradient_descent_frozen(
    [8.5, 0.65, 1.2], [0.1, 0.2, -0.1], 1.0, 0.01, 10, frozen=[] # same values as above and the inclusion of what is and isn't frozen
)

# Frozen = [0, 2] run balance is free
bal_w, bal_err, bal_hist = gradient_descent_frozen(
        [8.5, 0.65, 1.2], [0.1, 0.2, -0.1], 1.0, 0.01, 10, frozen=[0, 2] # same values as above and the inclusion of what is and isn't frozen
)

# Frozen = [0, 2] run breath is free
bre_w, bre_err, bre_hist = gradient_descent_frozen(
        [8.5, 0.65, 1.2], [0.1, 0.2, -0.1], 1.0, 0.01, 10, frozen=[0, 1] # same values as above and the inclusion of what is and isn't frozen
)

# Plot balance weight: Unfrozen vs Frozen=[0, 2]
plt.plot([w[1] for w in base_hist], label="balance unfrozen (alpha=0.01)")
plt.plot([w[1] for w in bal_hist], label = "balance frozen=[0,2] (alpha = 0.01)")

# Plot balance weight: Unfrozen vs Frozen=[0, 1]
plt.plot([w[2] for w in base_hist], label="balance unfrozen (alpha=0.01)")
plt.plot([w[2] for w in bre_hist], label = "balance frozen=[0,1] (alpha = 0.01)")

plt.xlabel("iteration")
plt.ylabel("weight value")
plt.title("Part 4 frozen vs unfrozen")
plt.legend()
plt.savefig("week4/fig2_frozen.png", dpi=150)
plt.close() 

""" 
Comment Block:

1. The weights in the frozen examples are steeper than the unfrozen examples. Not by much but enough to see that the line has grown steeper.

2. The blade angle weight is the only one that does get steeper at a glance so it is the steepest by default. It does match the part 1 prediction.

3. They are not. They only appear to be because of how little the incline is. To see a more defined incline or decline, we could plot the error contribution or the absolute change per iteration.
"""


### Part 6: Tests - Everyone

In [ ]:

#===================================== PART 1 & PART 1B==============================================

# example tests for Part 1 and Part 1b -- these get merged into the

from part1_multi_input import gradient_descent_multi
from part1b_rescaled import normalize

def test_normalize_max_becomes_one():
    # the biggest value in the channel should land exactly on 1.0
    result = normalize([8.5, 9.5, 9.9, 9.0])
    assert result[2] == 1.0

def test_normalize_does_not_modify_original():
    # normalize should return a NEW list, not change the one it was given
    original = [8.5, 9.5, 9.9, 9.0]
    original_copy = list(original)
    normalize(original)
    assert original == original_copy

def test_gradient_descent_multi_error_goes_down():
    # error on the last iteration should be smaller than on the first
    _, errors, _ = gradient_descent_multi([8.5, 0.65, 1.2], [0.1, 0.2, -0.1], 1.0, 0.01, 10)
    assert errors[-1] < errors[0]

def test_gradient_descent_multi_prediction_close_to_goal():
    # after enough iterations on an easy sensing, prediction should be near the goal
    final_weights, _, _ = gradient_descent_multi([2.0, 1.0, 3.0], [0.2, 0.5, -0.1], 1.0, 0.01, 200)
    pred = sum(w * i for w, i in zip(final_weights, [2.0, 1.0, 3.0]))
    assert abs(pred - 1.0) < 0.01

if __name__ == "__main__":
    tests = [
        test_normalize_max_becomes_one,
        test_normalize_does_not_modify_original,
        test_gradient_descent_multi_error_goes_down,
        test_gradient_descent_multi_prediction_close_to_goal,
    ]
    for test in tests:
        try:
            test()
            print(f"PASS: {test.__name__}")
        except AssertionError:
            print(f"FAIL: {test.__name__}")

#========================================================================================================

#===================================== PART 2 ===========================================================

from part2_multi_output import gradient_descent_outputs

def test_predictions_move_closer_to_targets():
    input = .65
    weights = [.3, .2, .9]
    trues = [0, 1, 0]

    initial_predictions = [input * w for w in weights]

    final_weights, error_history, weight_history = gradient_descent_outputs(input, weights, trues, .1, 10)

    final_predictions = [input * w for w in final_weights]

    for i in range(3):
        initial_error = abs(initial_predictions[i] - trues[i])
        final_error = abs(final_predictions[i] - trues[i])

        assert final_error < initial_error
        


def test_weights_change_when_predictions_wrong():
    input = .65
    starting_weights = [.3, .2, .9]
    trues = [0, 1, 0]

    final_weights, error_history, weight_history = gradient_descent_outputs(input, starting_weights.copy(), trues, .1, 1)

    for i in range(3):
        assert final_weights[i] != starting_weights[i]

#========================================================================================================

#===================================== PART 3 ===========================================================

from part3_multi_in_out import outer_prod

def test_outer_prod():
    result = outer_prod([1, 2], [3, 4, 5])

    assert result == [[3, 4, 5],
                      [6, 8, 10]]



#========================================================================================================

#===================================== PART 4 ===========================================================
from part4_freeze import gradient_descent_frozen


def test_frozen():
    inputs = [[.5, .7, .8], [1, 1, 1]]
    trues = [1, -.5]
    starting_ws = [1, 1, 1]
    frozen_indices = [[1, 2], [0]]
    unfrozen_indices = [[0], [1,2]]
    for i in range(2):
        final_ws, errors, weights = gradient_descent_frozen(inputs[i], starting_ws, trues[i], 1, 5, frozen_indices[i])
        for index in frozen_indices[i]:
            assert starting_ws[index] == final_ws[index]
        for index in unfrozen_indices[i]:
            assert starting_ws[index] != final_ws[index]
        



#========================================================================================================

#===================================== PART 5 ===========================================================

if __name__ == '__main__':
    tests = [name for name in dir() if name.startswith('test_')]
    passed = []
    failed = []
    for test_name in sorted(tests):
        test_func = globals()[test_name]
        try:
            test_func()
            passed.append(test_name)
            print(f' PASS: {test_name}')
        except AssertionError as e:
            failed.append(test_name)
            print(f' FAIL: {test_name} -- {e}')
    for name in passed:
        print(f'PASS: {name}')
    for name in failed:
        print(f'FAIL: {name}')
